# Dax Virani | Interactive Data Dashboard
A notebook-native version of the dashboard workflow with theme-aware exploration, smart column detection, missing-value handling, and reusable Plotly chart builders.

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_PATH = Path("dashboard_data.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    rng = np.random.default_rng(42)
    n = 420
    df = pd.DataFrame({
        "Region": rng.choice(["North", "South", "East", "West"], n),
        "Segment": rng.choice(["Consumer", "Corporate", "Home Office"], n),
        "Channel": rng.choice(["Web", "Retail", "Direct"], n),
        "Sales": rng.normal(2400, 900, n).clip(120, 6000).round(2),
        "Profit": rng.normal(260, 420, n).round(2),
        "Discount": rng.choice([0, 0, 0.05, 0.1, 0.15, 0.2], n),
        "Order Date": pd.date_range("2022-01-01", periods=n, freq="D")
    })
    df.to_csv(DATA_PATH, index=False)

df.head()

,Region,Segment,Channel,Sales,Profit,Discount,Order Date
0,North,Home Office,Web,2887.02,106.94,0.1,2022-01-01
1,West,Home Office,Retail,3086.63,-655.69,0.0,2022-01-02
2,East,Home Office,Direct,2803.29,275.15,0.1,2022-01-03
3,South,Corporate,Web,882.96,NaN,0.2,2022-01-04
4,South,Corporate,Web,2884.23,699.12,0.0,2022-01-05


## Data Hygiene and Column Profiling
The notebook infers numeric and categorical fields, then applies an adjustable missing-value strategy.

In [6]:
def detect_columns(frame):
    numeric = frame.select_dtypes(include="number").columns.tolist()
    categorical = frame.select_dtypes(exclude="number").columns.tolist()
    return numeric, categorical

def handle_missing(frame, strategy="median"):
    cleaned = frame.copy()
    numeric, _ = detect_columns(cleaned)
    if strategy == "drop":
        return cleaned.dropna()
    if strategy == "zero":
        cleaned[numeric] = cleaned[numeric].fillna(0)
    elif strategy == "median":
        for column in numeric:
            cleaned[column] = cleaned[column].fillna(cleaned[column].median())
    return cleaned

df_clean = handle_missing(df, "median")
numeric_cols, categorical_cols = detect_columns(df_clean)
summary = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "missing": df_clean.isna().sum(),
    "nunique": df_clean.nunique()
})
summary

,dtype,missing,nunique
Region,object,0,4
Segment,object,0,3
Channel,object,0,3
Sales,float64,0,417
Profit,float64,0,402
Discount,float64,0,5
Order Date,object,0,420


## Feature-Rich Exploration
This section builds the same style of interactive visuals the Streamlit version used, but in a notebook-friendly format.

In [ ]:
%pip install statsmodels

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.5 MB 18.3 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.5 MB 7.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.5 MB 7.7 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.5 MB 8.1 MB/s eta 0:00:01
   ----------------------------------- ---- 8.4/9.5 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 6.0 MB/s  0:00:01

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ----------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
fig1 = px.histogram(df_clean, x=numeric_cols[0], color=categorical_cols[0], nbins=30, title=f"Distribution of {numeric_cols[0]}")
fig2 = px.scatter(df_clean, x="Sales", y="Profit", color="Segment", size="Discount", trendline="ols", title="Sales vs Profit")
fig3 = px.box(df_clean, x="Region", y="Sales", color="Segment", points="outliers", title="Sales spread by Region")

for fig in [fig1, fig2, fig3]:
    fig.update_layout(template="plotly_white", height=460, margin=dict(l=20, r=20, t=60, b=20))
    fig.show()

## Executive Snapshot
A compact summary block that surfaces the core health and trend signals from the loaded dataset.

In [8]:
insights = pd.Series({
    "rows": len(df_clean),
    "columns": df_clean.shape[1],
    "missing_values": int(df_clean.isna().sum().sum()),
    "sales_mean": round(df_clean["Sales"].mean(), 2),
    "profit_mean": round(df_clean["Profit"].mean(), 2),
    "profit_median": round(df_clean["Profit"].median(), 2),
})
display(insights.to_frame("value"))
display(df_clean.describe(include="all").transpose().head(10))

,value
rows,420.00
columns,7.00
missing_values,0.00
sales_mean,2368.65
profit_mean,260.09
profit_median,262.58


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,420,4,South,115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,420,3,Home Office,154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Channel,420,3,Retail,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales,420.0,NaN,NaN,NaN,2368.645405,894.100768,120.0,1772.745,2411.72,2936.375,5260.97
Profit,420.0,NaN,NaN,NaN,260.085881,426.064045,-981.36,-11.2825,262.58,535.2975,1439.87
Discount,420.0,NaN,NaN,NaN,0.082738,0.073966,0.0,0.0,0.05,0.15,0.2
Order Date,420,420,2022-01-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
